# Notebook 01 — `SELECT`, Filtering (`WHERE`, `IN`, `BETWEEN`, `IS NULL`), and Sorting (`ORDER BY`, `LIMIT`)

**Session:** Database - SQL (1)  
**Module:** 02 · Calculus and Linear Algebra

## Goal

Load the `sql_store` schema from `../data/sql_store_seed.sql` into an in-memory SQLite database and practice foundational single-table SQL queries: column projection, arithmetic expressions, aliases, `DISTINCT`, comparison and logical operators, `IN`, `BETWEEN`, `IS NULL`, `ORDER BY`, and `LIMIT`/`OFFSET`.

In [ ]:
import sqlite3
from pathlib import Path

SEED_PATH = Path("../data/sql_store_seed.sql")
conn = sqlite3.connect(":memory:")
conn.executescript(SEED_PATH.read_text(encoding="utf-8"))


def query(sql: str) -> list[dict]:
    """Execute a SQL statement and return rows as a list of dictionaries."""
    cur = conn.execute(sql)
    cols = [desc[0] for desc in cur.description]
    return [dict(zip(cols, row, strict=True)) for row in cur.fetchall()]


def show_rows(rows: list[dict], limit: int | None = None) -> None:
    """Pretty-print query results as an aligned ASCII table."""
    display_rows = rows if limit is None else rows[:limit]
    if not display_rows:
        print("(0 rows)")
        return
    cols = list(display_rows[0].keys())
    widths = {c: max(len(c), *(len(str(r[c])) for r in display_rows)) for c in cols}
    header = " | ".join(f"{c:<{widths[c]}}" for c in cols)
    sep = "-+-".join("-" * widths[c] for c in cols)
    print(header)
    print(sep)
    for r in display_rows:
        print(" | ".join(f"{r[c]!s:<{widths[c]}}" for c in cols))


print("Tables available:")
show_rows(query("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name;"))

## 1. `SELECT`, Arithmetic Expressions, Aliases (`AS`), and `DISTINCT`

In [ ]:
# Select specific columns
show_rows(query("SELECT first_name, last_name FROM customers;"), limit=5)

# Arithmetic expressions and column aliases
show_rows(
    query(
        """
        SELECT
            customer_id,
            points,
            points + 10 AS points_plus_ten,
            ROUND(points * 1.10, 2) AS points_with_vat
        FROM customers;
        """
    ),
    limit=5,
)

# Compute a 10% price increase on products
show_rows(
    query(
        """
        SELECT
            name,
            unit_price,
            ROUND(unit_price * 1.1, 2) AS new_price
        FROM products;
        """
    ),
    limit=5,
)

# Distinct customer states
states = [r["state"] for r in query("SELECT DISTINCT state FROM customers ORDER BY state;")]
print("Distinct states:", states)

## 2. Filtering Rows with `WHERE`, Comparison, and Logical Operators

In [ ]:
# High-point customers
show_rows(query("SELECT customer_id, first_name, points FROM customers WHERE points > 3000;"))

# Equality vs inequality
show_rows(query("SELECT customer_id, state FROM customers WHERE state = 'VA';"))
non_va = query("SELECT customer_id FROM customers WHERE state != 'VA';")
print("Non-VA count:", len(non_va))

# Date filtering (ISO-8601 strings compare chronologically)
show_rows(
    query("SELECT order_id, order_date FROM orders WHERE order_date >= '2018-01-01';"),
    limit=5,
)

# Arithmetic predicate in WHERE
order6 = query(
    """
    SELECT order_id, product_id, quantity, unit_price, ROUND(unit_price * quantity, 2) AS total_price
    FROM order_items
    WHERE order_id = 6 AND unit_price * quantity < 30;
    """
)
show_rows(order6)

## 3. `IN`, `BETWEEN`, `IS NULL`, `ORDER BY`, and `LIMIT`

In [ ]:
# IN vs OR chains, and BETWEEN (inclusive bounds)
region_rows = query(
    """
    SELECT customer_id, first_name, state, points
    FROM customers
    WHERE state IN ('VA', 'GA', 'FL')
      AND points BETWEEN 300 AND 2000;
    """
)
show_rows(region_rows)

# Null checks (IS NULL / IS NOT NULL)
no_phone = query("SELECT customer_id, first_name, last_name FROM customers WHERE phone IS NULL;")
print("Customers missing phone:", len(no_phone))
assert len(no_phone) == 2

# Sorting and pagination (LIMIT count OFFSET skip)
top3 = query("SELECT customer_id, first_name, points FROM customers ORDER BY points DESC LIMIT 3;")
next4 = query(
    "SELECT customer_id, first_name, points FROM customers ORDER BY points DESC LIMIT 4 OFFSET 3;"
)
print("Top 3:")
show_rows(top3)
print("Ranks 4-7:")
show_rows(next4)
assert {r["customer_id"] for r in top3}.isdisjoint({r["customer_id"] for r in next4})